# Probability Fundamentals

| Difficulty | ██░░░░░░░░ 2/10 |
| :--- | :--- |
| Prerequisites | 00.01–00.04 |
| Time | ~55 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/03_Probability_and_Statistics/01_probability_fundamentals.ipynb)

---

## 🎯 Learning Objective

Formalize probability through sample spaces, events, and axioms, and learn to compute conditional probabilities.

---

## 📐 Theory

Probability theory gives us a precise language for reasoning about uncertainty — exactly what
we need before any model can quantify "how confident" it is, or how likely a dataset is under
a hypothesis. Everything in this module builds on four ideas: sample spaces, events, the axioms
a valid probability must satisfy, and conditional probability.

### Sample spaces and events

The **sample space** $\Omega$ is the set of every possible outcome of a random experiment.
Rolling a die gives $\Omega=\{1,2,3,4,5,6\}$; a coin flip gives $\Omega=\{H,T\}$. An **event**
$A$ is just a subset of $\Omega$ — "the die shows an even number" is the event
$A=\{2,4,6\}\subseteq\Omega$. Events are sets, so everything from `00.02` (unions,
intersections, complements) applies directly: $A\cup B$ ("A or B"), $A\cap B$ ("A and B"),
$A^c$ ("not A").

### The axioms of probability

A **probability measure** $P$ assigns a number to every event, and is valid if and only if it
satisfies Kolmogorov's three axioms:

1. **Non-negativity**: $P(A) \ge 0$ for every event $A$.
2. **Normalization**: $P(\Omega) = 1$ — something in the sample space always happens.
3. **Countable additivity**: if $A_1, A_2, \ldots$ are pairwise disjoint ($A_i \cap A_j =
   \emptyset$ for $i \ne j$), then $P\!\left(\bigcup_i A_i\right) = \sum_i P(A_i)$.

Every other rule you've ever used to compute a probability is a *theorem* derived from these
three axioms, not an extra assumption. The **complement rule** $P(A^c) = 1 - P(A)$ follows
immediately from axioms 2 and 3 applied to the disjoint pair $A, A^c$. The **addition rule**
for overlapping events,

$$P(A \cup B) = P(A) + P(B) - P(A \cap B),$$

corrects the double-counting of $A\cap B$ that naive additivity would produce; it collapses to
axiom 3 exactly when $A \cap B = \emptyset$.

### Counting: probability on equally-likely outcomes

When $\Omega$ has $n$ equally likely outcomes, $P(A) = |A|/|\Omega|$ — probability becomes
counting. Two counting rules recur constantly: the number of ways to *order* $k$ items chosen
from $n$ (**permutations**) is $n!/(n-k)!$, and the number of ways to *choose* $k$ items from
$n$ without regard to order (**combinations**) is $\binom{n}{k} = n!/(k!(n-k)!)$. Combinations
reappear the moment we ask for the probability of "exactly $k$ successes out of $n$ trials" —
the binomial distribution of [A Gallery of Common Distributions](06_common_distributions_gallery.ipynb) (`03.06`).

### Conditional probability and independence

Learning that event $B$ occurred should update your belief about $A$. The **conditional
probability** of $A$ given $B$ is

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}, \qquad P(B) > 0,$$

the fraction of $B$'s probability that also lies in $A$ — literally *restricting the sample
space to $B$* and renormalizing. Rearranging gives the **multiplication rule** $P(A\cap B) =
P(A\mid B)P(B)$, which chains: $P(A\cap B\cap C) = P(A\mid B,C)\,P(B\mid C)\,P(C)$. This chain
rule of probability is the identity that lets [Joint, Conditional, and Marginal Distributions](04_joint_conditional_marginal.ipynb) (`03.04`) factor a joint distribution into conditionals.

Two events are **independent** when knowing $B$ tells you nothing about $A$: $P(A\mid B) =
P(A)$, equivalently $P(A \cap B) = P(A)P(B)$. Independence is an assumption you impose on a
model, not something you get for free — deciding *which* independence assumptions are
reasonable is much of the work of building a probabilistic model.

Finally, the **law of total probability**: if $B_1,\ldots,B_k$ partition $\Omega$ (disjoint,
covering everything), then $P(A) = \sum_i P(A\mid B_i)P(B_i)$ — an unconditional probability
can always be computed by conditioning on every way the world could be and weighting by how
likely each way is. This is exactly the identity that makes [Bayes' Theorem and Inference](05_bayes_theorem_and_inference.ipynb) (`03.05`) computable.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from math import comb, factorial
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

The sample space of rolling two dice has 36 equally-likely outcomes. Coloring in two events on
that grid makes "event = subset" and "conditional probability = restrict and renormalize"
visible at a glance, instead of abstract.

In [ ]:
# Sample space of two fair dice: 36 equally likely outcomes (first_die, second_die).
# Event A = "the two dice sum to 7", Event B = "the first die shows 3".
outcomes = list(product(range(1, 7), range(1, 7)))
event_A = {(d1, d2) for d1, d2 in outcomes if d1 + d2 == 7}
event_B = {(d1, d2) for d1, d2 in outcomes if d1 == 3}
intersection = event_A & event_B

fig, ax = plt.subplots(figsize=(6, 6))
for d1, d2 in outcomes:
    if (d1, d2) in intersection:
        color = "#C44E52"    # in BOTH A and B
    elif (d1, d2) in event_A:
        color = "#4C72B0"    # in A only
    elif (d1, d2) in event_B:
        color = "#DD8452"    # in B only
    else:
        color = "#DDDDDD"    # neither
    ax.add_patch(plt.Rectangle((d1 - 0.5, d2 - 0.5), 1, 1, facecolor=color, edgecolor="white"))
ax.set_xlim(0.5, 6.5); ax.set_ylim(0.5, 6.5)
ax.set_xticks(range(1, 7)); ax.set_yticks(range(1, 7))
ax.set_xlabel("first die"); ax.set_ylabel("second die")
ax.set_aspect("equal")
ax.set_title("A = {sum=7} (blue), B = {first die=3} (orange), A∩B (red)")
plt.tight_layout()
plt.show()

print(f"|Omega| = {len(outcomes)}")
print(f"P(A) = {len(event_A)}/{len(outcomes)} = {len(event_A)/len(outcomes):.4f}")
print(f"P(B) = {len(event_B)}/{len(outcomes)} = {len(event_B)/len(outcomes):.4f}")
print(f"P(A and B) = {len(intersection)}/{len(outcomes)} = {len(intersection)/len(outcomes):.4f}")
print(f"P(A | B) = P(A and B)/P(B) = {len(intersection)/len(event_B):.4f}  "
      f"-- within row B (orange+red), what fraction is also red?")

## 🐍 Implementation from Scratch

We implement permutations and combinations from first principles, then verify all three axioms
and the conditional-probability definition directly on a concrete sample space — nothing here
is assumed, everything is checked by counting.

In [ ]:
# From-scratch counting: permutations and combinations, verified against math.comb
def permutations_count(n, k):
    return factorial(n) // factorial(n - k)

def combinations_count(n, k):
    return factorial(n) // (factorial(k) * factorial(n - k))

for n, k in [(6, 2), (10, 3), (52, 5)]:
    assert permutations_count(n, k) == factorial(n) // factorial(n - k)
    assert combinations_count(n, k) == comb(n, k)
print("permutations_count/combinations_count match Python's math.comb for (6,2), (10,3), (52,5)")
print(f"e.g. number of distinct 5-card poker hands from a 52-card deck: C(52,5) = {combinations_count(52, 5):,}")

# Verify the THREE axioms directly on the two-dice sample space -- not assumed, checked
omega = list(product(range(1, 7), range(1, 7)))
p = {outcome: 1 / len(omega) for outcome in omega}  # uniform probability measure

print(f"\nAxiom 1 (non-negativity): all P(outcome) >= 0? {all(v >= 0 for v in p.values())}")
print(f"Axiom 2 (normalization): P(Omega) = sum of every outcome's probability = {sum(p.values()):.6f}")

sum_is_7 = {o for o in omega if o[0] + o[1] == 7}
sum_is_2 = {o for o in omega if o[0] + o[1] == 2}   # only (1,1) -- disjoint from sum=7
union_prob = sum(p[o] for o in (sum_is_7 | sum_is_2))
additive_prob = sum(p[o] for o in sum_is_7) + sum(p[o] for o in sum_is_2)
print(f"Axiom 3 (additivity): P(sum=7 or sum=2) = {union_prob:.6f}, "
      f"P(sum=7)+P(sum=2) = {additive_prob:.6f}  (equal, because the events are disjoint)")

# Conditional probability FROM its definition (counting): restricting the sample space and
# renormalizing must give the identical number as P(A and B)/P(B) -- verify, don't assume
event_A = {o for o in omega if o[0] + o[1] == 7}
event_B = {o for o in omega if o[0] == 3}
p_A_given_B_by_restriction = len(event_A & event_B) / len(event_B)
p_A_given_B_by_formula = (len(event_A & event_B) / len(omega)) / (len(event_B) / len(omega))
print(f"\nP(A|B) by restricting to B:      {p_A_given_B_by_restriction:.6f}")
print(f"P(A|B) by formula P(A∩B)/P(B):   {p_A_given_B_by_formula:.6f}  (identical, as the definition guarantees)")

# Law of total probability: partition Omega by the first die's value, sum P(A|B_i) P(B_i)
total_prob_check = sum(
    (len(event_A & {o for o in omega if o[0] == first}) / 6) * (6 / 36)
    for first in range(1, 7)
)
print(f"\nLaw of total probability: sum_i P(A | first_die=i) P(first_die=i) = {total_prob_check:.6f}")
print(f"Direct count P(A) = {len(event_A) / len(omega):.6f}  (matches)")

## 🤖 Why This Matters for AI

**Data likelihood** — how probable an observed dataset is under a hypothesis — underlies every
probabilistic model, and it's built entirely from the axioms above: independence gives you a
product rule over data points, and counting gives you the coefficient in front. **Quantifying
model uncertainty** works the same way: a classifier's raw accuracy number hides the question
you actually care about, and conditional probability answers it directly.

---

In [ ]:
# "Data likelihood": which hypothesis about a spam rate makes the OBSERVED data most probable?
# Independence (an axiom-derived product rule) lets P(data | hypothesis) factor cleanly.
n_emails, n_spam_observed = 20, 6   # observed: 6 of 20 incoming emails were spam

candidate_rates = [0.10, 0.20, 0.30, 0.50, 0.70]
print("Likelihood of observing 6/20 spam emails, under different hypothesized spam rates p:")


def likelihood(p, n=n_emails, k=n_spam_observed):
    # P(exactly k spam out of n, each iid) = C(n,k) p^k (1-p)^(n-k) -- counting * independence
    return comb(n, k) * p**k * (1 - p)**(n - k)


for p in candidate_rates:
    print(f"  p_spam={p:.1f}: P(data | p_spam) = {likelihood(p):.6f}")
best_p = max(candidate_rates, key=likelihood)
print(f"Most likely hypothesis among candidates: p_spam={best_p:.1f} "
      f"-- exactly the observed rate {n_spam_observed/n_emails:.2f} (formalized as MLE in 03.07)")

# "Model uncertainty": a spam filter's headline accuracy numbers are misleading on their own --
# conditional probability answers what you actually want: given a flagged email, how likely is
# it truly spam?
true_positive_rate = 0.95     # P(flagged | actually spam)      -- filter catches 95% of spam
false_positive_rate = 0.03    # P(flagged | actually not spam)  -- filter misflags 3% of real mail
p_spam_base_rate = 0.20       # P(actually spam) -- share of incoming mail that is truly spam

# Law of total probability: P(flagged) = P(flagged|spam)P(spam) + P(flagged|not spam)P(not spam)
p_flagged = true_positive_rate * p_spam_base_rate + false_positive_rate * (1 - p_spam_base_rate)
# Conditional probability restricted to the "flagged" world: what fraction of it is truly spam?
p_spam_given_flagged = (true_positive_rate * p_spam_base_rate) / p_flagged

print(f"\nFilter catches {true_positive_rate:.0%} of spam, misflags {false_positive_rate:.0%} of real mail.")
print(f"P(flagged) = {p_flagged:.4f}")
print(f"P(actually spam | flagged) = {p_spam_given_flagged:.4f}  "
      f"-- meaningfully different from the {true_positive_rate:.0%} headline accuracy number")
print("(this algebra -- conditioning plus the law of total probability -- IS Bayes' theorem,")
print(" generalized and named explicitly in 03.05)")

## 🏋️ Exercises

### Warm-up
1. By hand, use the complement rule to compute $P(\text{at least one die shows a 6})$ when
   rolling two fair dice. Verify your answer with code, using the `omega` sample space above.

### Practice
2. Pick any event and any partition of the two-dice sample space (e.g. partition by the value
   of the second die instead of the first). Estimate $P(A)$ two ways: (a) exact counting, and
   (b) simulating 100,000 rolls with `np.random.randint` and computing the empirical frequency.
   Confirm they agree to roughly 2-3 decimal places.

### Challenge
3. A classifier is independently correct with probability $0.9$ on each of 10 test examples.
   Using combinations, compute the probability it gets *at least* 8 of the 10 correct. Then
   look up how this relates to the binomial distribution you'll meet formally in `03.06`, and
   explain in one sentence why "independently correct" is the assumption doing all the work.

---

## 📚 Further Reading
- Blitzstein & Hwang, *Introduction to Probability* (Harvard Stat 110)
- Grinstead & Snell, [*Introduction to Probability*](https://math.dartmouth.edu/~prob/prob/prob.pdf) (free online)
- Ross, *A First Course in Probability*

---

*Next notebook: [Random Variables and Distributions](02_random_variables_distributions.ipynb)*